# NB5 — Extension: can RL be made to *earn* its place?

Three targeted changes to the original design, tested on a reduced but tier-balanced scope.

| # | Change | Fixes |
|---|---|---|
| 1 | **Differential Sharpe Ratio reward** (Moody & Saffell 1998) | agent now optimises what we measure |
| 2 | **Fine exposure grid** (11 levels, was 3) | can express real volatility scaling |
| 3 | **Residual RL** — agent learns a correction *on top of* the vol-target rule | structurally answers "why RL?" |

**Two arms**
* **Arm A (effect size):** standard exposure actions + DSR reward, 3 state specs.
* **Arm B (benchmark):** residual over the VT rule — can RL improve on the rule it starts from?

Scope: 6 markets (all four tiers) x 2 agents (Q-Learning, PPO) x 5 seeds.
Checkpointed to Drive exactly like NB4 — safe to disconnect and resume.

In [1]:
# ============ CELL 0 — setup ============
!pip -q install yfinance 2>/dev/null
import os, math, random, time, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive"); DRIVE="/content/drive/MyDrive"
except Exception: DRIVE="."
OUT=os.path.join(DRIVE,"nb4_outputs")           # reuse the same folder
CKPT=os.path.join(OUT,"checkpoints")            # NB4 price/signal caches live here
EXT=os.path.join(OUT,"ext_checkpoints"); os.makedirs(EXT,exist_ok=True)

import torch, torch.nn as nn
DEV="cuda" if torch.cuda.is_available() else "cpu"

CFG=dict(
  vol_window=5, train_frac=0.70, val_frac=0.15,
  agents=["Q-Learning","PPO"],
  states=["Baseline","+RetFore","+VolFore"],
  seeds=[0,1,2,3,4],
  episodes=300,
  initial_capital=10000.0, transaction_cost=0.001, slippage=0.0005,
  gamma=0.95, alpha=0.10,
  eps_start=1.0, eps_min=0.05, eps_decay=0.99,
  n_levels=11,                 # CHANGE 2: fine exposure grid (was 3)
  residual_deltas=[-0.3,-0.15,0.0,0.15,0.3],   # CHANGE 3: residual action set
  vol_target_annual=0.15, max_leverage=1.0,
  dsr_eta=0.02, dsr_warmup=30, # CHANGE 1: DSR memory + warm-up
)
# tier-balanced, deliberately includes markets where NB4 succeeded AND failed
MARKETS=["SP500","FTSE100","KOSPI","Nifty50","DSE","BTC"]
GROUP={"SP500":"Developed","FTSE100":"Developed","KOSPI":"Emerging",
       "Nifty50":"Emerging","DSE":"Frontier","BTC":"Crypto"}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
print("device:",DEV,"| markets:",MARKETS,"| levels:",CFG["n_levels"])

Mounted at /content/drive
device: cpu | markets: ['SP500', 'FTSE100', 'KOSPI', 'Nifty50', 'DSE', 'BTC'] | levels: 11


In [2]:
# ============ CELL 0 — setup ============
!pip -q install yfinance 2>/dev/null
import os, math, random, time, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive"); DRIVE="/content/drive/MyDrive"
except Exception: DRIVE="."
OUT=os.path.join(DRIVE,"nb4_outputs")           # reuse the same folder
CKPT=os.path.join(OUT,"checkpoints")            # NB4 price/signal caches live here
EXT=os.path.join(OUT,"ext_checkpoints"); os.makedirs(EXT,exist_ok=True)

import torch, torch.nn as nn
DEV="cuda" if torch.cuda.is_available() else "cpu"

CFG=dict(
  vol_window=5, train_frac=0.70, val_frac=0.15,
  agents=["Q-Learning","PPO"],
  states=["Baseline","+RetFore","+VolFore"],
  seeds=[0,1,2,3,4],
  episodes=300,
  initial_capital=10000.0, transaction_cost=0.001, slippage=0.0005,
  gamma=0.95, alpha=0.10,
  eps_start=1.0, eps_min=0.05, eps_decay=0.99,
  n_levels=11,                 # CHANGE 2: fine exposure grid (was 3)
  residual_deltas=[-0.3,-0.15,0.0,0.15,0.3],   # CHANGE 3: residual action set
  vol_target_annual=0.15, max_leverage=1.0,
  dsr_eta=0.02, dsr_warmup=30, # CHANGE 1: DSR memory + warm-up
)
# ===== ALL 14 markets (same set + grouping as NB4) =====
MARKETS=["SP500","NASDAQ","FTSE100","Nikkei","DAX","ASX200",
         "Nifty50","Bovespa","KOSPI","MexIPC",
         "DSE","Vietnam","BTC","ETH"]
GROUP={"SP500":"Developed","NASDAQ":"Developed","FTSE100":"Developed",
       "Nikkei":"Developed","DAX":"Developed","ASX200":"Developed",
       "Nifty50":"Emerging","Bovespa":"Emerging","KOSPI":"Emerging","MexIPC":"Emerging",
       "DSE":"Frontier","Vietnam":"Frontier",
       "BTC":"Crypto","ETH":"Crypto"}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
print("device:",DEV,"| markets:",len(MARKETS),MARKETS,"| levels:",CFG["n_levels"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cpu | markets: 14 ['SP500', 'NASDAQ', 'FTSE100', 'Nikkei', 'DAX', 'ASX200', 'Nifty50', 'Bovespa', 'KOSPI', 'MexIPC', 'DSE', 'Vietnam', 'BTC', 'ETH'] | levels: 11


In [ ]:
# ---- run every market that isn't finished yet (both arms) ----
for m in MARKETS:
    # cache-check: NB5 needs NB4's price + signal cache for each market
    fp=os.path.join(CKPT,f"{m}_price.csv"); fs=os.path.join(CKPT,f"{m}_signals.npz")
    if not (os.path.exists(fp) and os.path.exists(fs)):
        print(f"[{m}] SKIP — missing NB4 cache ({os.path.basename(fp)}/{os.path.basename(fs)})")
        continue
    print("="*70,"\n",m)
    run_market(m, arm="A")   # effect size (the crossover)
    run_market(m, arm="B")   # residual vs VT rule
print("\nALL DONE — re-run the Analysis cell (Section 5).")

 SP500


NameError: name 'run_market' is not defined

## 1. Data + features (reuses the NB4 caches so numbers stay comparable)

In [ ]:
def rsi_(s,n=14):
    d=s.diff();g=d.clip(lower=0).rolling(n).mean();l=(-d.clip(upper=0)).rolling(n).mean()
    return (100-100/(1+g/l.replace(0,np.nan))).fillna(50.0)
RL_FEATS=["momentum","rsi","ma_diff","volatility"]
def build_features(px):
    h=CFG["vol_window"]; df=pd.DataFrame({"close":px}); df["log_ret"]=np.log(df["close"]).diff()
    df["ma_diff"]=(df["close"].rolling(5).mean()-df["close"].rolling(10).mean())/df["close"]
    df["rsi"]=rsi_(df["close"]); df["momentum"]=df["close"].pct_change(10)
    df["volatility"]=df["log_ret"].rolling(20).std()
    e12=df["close"].ewm(span=12,adjust=False).mean(); e26=df["close"].ewm(span=26,adjust=False).mean()
    df["macd"]=(e12-e26)/df["close"]
    mb=df["close"].rolling(20).mean(); sd=df["close"].rolling(20).std(); df["bb_pos"]=(df["close"]-mb)/(2*sd)
    rv=df["log_ret"].rolling(h).std()*math.sqrt(252)
    df["rv"]=rv; df["rv_d"]=rv; df["rv_w"]=rv.rolling(5).mean(); df["rv_m"]=rv.rolling(22).mean()
    df["target_ret"]=df["log_ret"].shift(-1); df["target_vol"]=rv.shift(-h)
    return df.dropna()

def load_market(m):
    """price + signals come from the NB4 checkpoints -> identical data, comparable results"""
    fp=os.path.join(CKPT,f"{m}_price.csv"); fs=os.path.join(CKPT,f"{m}_signals.npz")
    assert os.path.exists(fp) and os.path.exists(fs), f"missing NB4 cache for {m}"
    px=pd.read_csv(fp,parse_dates=["Date"]).set_index("Date")["Close"].astype(float).dropna()
    z=np.load(fs,allow_pickle=True)
    data=build_features(px)
    assert len(z["vol_signal"])==len(data), f"{m}: signal/feature length mismatch"
    return data,z["ret_signal"],z["vol_signal"],int(z["i_va"])
print("loaders ready")

loaders ready


## 2. Environment — DSR reward, fine grid, optional residual mode

In [ ]:
class DSR:
    """Differential Sharpe Ratio (Moody & Saffell 1998), with a warm-up so the
    variance estimate is established before the ratio is used. Without the warm-up
    the denominator (B-A^2)^1.5 is ~0 on the first steps and the reward saturates."""
    def __init__(self,eta,warmup=30): self.eta=eta; self.warmup=warmup; self.reset()
    def reset(self): self.A=0.0; self.B=0.0; self.k=0
    def __call__(self,r):
        self.k+=1
        if self.k<=self.warmup:                    # accumulate simple moments only
            w=1.0/self.k
            self.A+=w*(r-self.A); self.B+=w*(r*r-self.B)
            return 0.0
        dA=r-self.A; dB=r*r-self.B
        var=max(self.B-self.A**2, 1e-8)            # floored variance
        d=(self.B*dA-0.5*self.A*dB)/(var**1.5)
        self.A+=self.eta*dA; self.B+=self.eta*dB
        return float(np.clip(d,-5.0,5.0))

class Env:
    def __init__(self, df, kind, cfg, mu, sd, ret_sig, vol_sig, ret_sd, vmu, vsd,
                 residual=False, reward="dsr"):
        self.px=df["close"].values.astype(float)
        self.f=df[RL_FEATS].values.astype(float)
        self.ret=np.asarray(ret_sig,float); self.vol=np.asarray(vol_sig,float)
        self.kind=kind; self.cfg=cfg; self.mu=mu; self.sd=sd
        self.ret_sd=ret_sd; self.vmu=vmu; self.vsd=vsd
        self.residual=residual; self.reward_kind=reward
        self.n=len(self.px)
        self.EXPO=np.linspace(0.0,cfg["max_leverage"],cfg["n_levels"])
        self.DELTA=np.array(cfg["residual_deltas"],float)
        # base exposure from the volatility-target rule (used in residual mode)
        self.vt=np.clip(cfg["vol_target_annual"]/(self.vol+1e-9),0,cfg["max_leverage"])
        self.dsr=DSR(cfg["dsr_eta"],cfg.get("dsr_warmup",30))
    @property
    def n_actions(self): return len(self.DELTA) if self.residual else len(self.EXPO)
    @property
    def state_dim(self):  return len(RL_FEATS)+(0 if self.kind=="Baseline" else 1)+1
    def reset(self):
        self.t=0; self.cash=self.cfg["initial_capital"]; self.units=0.0; self.expo=0.0
        self.worth=[self.cash]; self.dsr.reset(); return self._state()
    def _state(self):
        s=list((self.f[self.t]-self.mu)/self.sd)
        if self.kind=="+RetFore":   s.append(self.ret[self.t]/(self.ret_sd+1e-12))
        elif self.kind=="+VolFore": s.append((self.vol[self.t]-self.vmu)/(self.vsd+1e-12))
        s.append(self.expo)
        return np.array(s,dtype=np.float32)
    def step(self,a):
        tgt=(np.clip(self.vt[self.t]+self.DELTA[a],0,self.cfg["max_leverage"])
             if self.residual else self.EXPO[a])
        p=self.px[self.t]; w=self.cash+self.units*p
        des=tgt*w/p; du=des-self.units
        cost=abs(du)*p*(self.cfg["transaction_cost"]+self.cfg["slippage"])
        self.cash-=du*p+cost; self.units=des; self.expo=tgt
        self.t+=1; done=self.t>=self.n-1
        w2=self.cash+self.units*self.px[self.t]
        ret=(w2-self.worth[-1])/max(self.worth[-1],1e-9)
        self.worth.append(w2)
        r=self.dsr(ret) if self.reward_kind=="dsr" else (w2-self.worth[-2])/self.cfg["initial_capital"]
        return self._state(), r, done

def perf(w,periods=252):
    w=np.asarray(w,float); r=np.diff(w)/w[:-1]; ann=math.sqrt(periods)
    sh=r.mean()/(r.std()+1e-12)*ann
    dsd=r[r<0].std() if (r<0).any() else 1e-12
    peak=np.maximum.accumulate(w); mdd=((w-peak)/peak).min()
    yrs=len(w)/periods; cagr=(w[-1]/w[0])**(1/yrs)-1
    return dict(NetWorth=float(w[-1]),Sharpe=float(sh),Sortino=float(r.mean()/(dsd+1e-12)*ann),
                MaxDD=float(mdd),CAGR=float(cagr),Calmar=float(cagr/(abs(mdd)+1e-12)))
print("env ready (DSR reward, %d exposure levels, residual mode available)"%CFG["n_levels"])

env ready (DSR reward, 11 exposure levels, residual mode available)


## 3. Agents (Q-Learning + PPO, action count now variable)

In [ ]:
class Disc:
    def __init__(self,kind):
        self.edges=[np.array([-1.,0.,1.]),np.array([-.5,.5]),np.array([-1.,0.,1.]),np.array([-.5,.5])]
        if kind!="Baseline": self.edges.append(np.array([-.5,.5]))
        self.dims=[len(e)+1 for e in self.edges]+[3]     # exposure kept coarse (3 bins)
    def __call__(self,s):
        idx=[int(np.digitize(s[i],e)) for i,e in enumerate(self.edges)]
        idx.append(int(np.clip(round(s[-1]*2),0,2))); return tuple(idx)

def train_q(env,cfg,seed):
    set_seed(seed); disc=Disc(env.kind); nA=env.n_actions
    Q=np.zeros(tuple(disc.dims)+(nA,)); eps=cfg["eps_start"]
    for _ in range(cfg["episodes"]):
        s=disc(env.reset()); done=False
        while not done:
            a=np.random.randint(nA) if np.random.rand()<eps else int(Q[s].argmax())
            s2r,r,done=env.step(a); s2=disc(s2r)
            Q[s][a]+=cfg["alpha"]*(r+cfg["gamma"]*Q[s2].max()*(not done)-Q[s][a]); s=s2
        eps=max(cfg["eps_min"],eps*cfg["eps_decay"])
    return lambda st:int(Q[disc(st)].argmax())

class AC(nn.Module):
    def __init__(self,di,nA):
        super().__init__(); self.b=nn.Sequential(nn.Linear(di,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh())
        self.pi=nn.Linear(64,nA); self.v=nn.Linear(64,1)
    def forward(self,x):
        h=self.b(x); return torch.distributions.Categorical(logits=self.pi(h)), self.v(h).squeeze(-1)

def train_ppo(env,cfg,seed,clip=0.2,lam=0.95,epochs=4):
    set_seed(seed); ac=AC(env.state_dim,env.n_actions).to(DEV)
    opt=torch.optim.Adam(ac.parameters(),lr=3e-4)
    for _ in range(cfg["episodes"]):
        S,A,R,LP,V=[],[],[],[],[]; s=env.reset(); done=False
        while not done:
            st=torch.tensor(s[None]).to(DEV)
            with torch.no_grad(): dist,v=ac(st); a=dist.sample()
            s2,r,done=env.step(int(a))
            S.append(s);A.append(int(a));R.append(r);LP.append(float(dist.log_prob(a)));V.append(float(v));s=s2
        adv=np.zeros(len(R)); g=0.0
        for t in reversed(range(len(R))):
            nv=V[t+1] if t+1<len(R) else 0.0
            g=(R[t]+cfg["gamma"]*nv-V[t])+cfg["gamma"]*lam*g; adv[t]=g
        ret=adv+np.array(V); adv=(adv-adv.mean())/(adv.std()+1e-8)
        S=torch.tensor(np.array(S),dtype=torch.float32).to(DEV); A=torch.tensor(A).long().to(DEV)
        LP=torch.tensor(LP,dtype=torch.float32).to(DEV); AD=torch.tensor(adv,dtype=torch.float32).to(DEV)
        RT=torch.tensor(ret,dtype=torch.float32).to(DEV)
        for _ in range(epochs):
            dist,v=ac(S); ratio=torch.exp(dist.log_prob(A)-LP)
            pl=-torch.min(ratio*AD,torch.clamp(ratio,1-clip,1+clip)*AD).mean()
            loss=pl+0.5*nn.functional.mse_loss(v,RT)-0.01*dist.entropy().mean()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(ac.parameters(),0.5); opt.step()
    return lambda st:int(ac(torch.tensor(st[None]).to(DEV))[0].probs.argmax())

TRAIN={"Q-Learning":train_q,"PPO":train_ppo}
def rollout(pol,env):
    s=env.reset(); done=False
    while not done: s,_,done=env.step(pol(s))
    return np.array(env.worth)
print("agents ready")

agents ready


## 4. Runner — Arm A (effect size) and Arm B (residual vs benchmark)

In [ ]:
def run_market(m, arm="A"):
    """arm A: standard exposure actions, 3 states.  arm B: residual over VT rule."""
    path=os.path.join(EXT,f"{m}_arm{arm}.csv"); rows=[]; done=set()
    if os.path.exists(path):
        prev=pd.read_csv(path); rows=prev.to_dict("records")
        done={(r["agent"],r["state"],int(r["seed"])) for r in rows}
        print(f"[{m}/{arm}] resume: {len(rows)} rows")
    states=CFG["states"] if arm=="A" else ["Baseline","+VolFore"]
    total=len(CFG["agents"])*len(states)*len(CFG["seeds"])+1
    if len(done)>=total: print(f"[{m}/{arm}] complete, skip"); return
    data,ret_sig,vol_sig,iva=load_market(m)
    tr,te=data.iloc[:iva],data.iloc[iva:]
    mu=tr[RL_FEATS].values.mean(0); sd=tr[RL_FEATS].values.std(0)+1e-12
    rsd=np.nanstd(ret_sig[:iva]); vmu=np.nanmean(vol_sig[:iva]); vsd=np.nanstd(vol_sig[:iva])
    def mk(df,kind,sl,residual):
        return Env(df,kind,CFG,mu,sd,ret_sig[sl],vol_sig[sl],rsd,vmu,vsd,
                   residual=residual,reward="dsr")
    save=lambda: pd.DataFrame(rows).to_csv(path,index=False)
    # reference row: the pure VT rule on the same test window
    if ("VTrule","-",-1) not in done:
        e=mk(te,"+VolFore",slice(iva,None),False)
        e.reset(); w=[e.cash]
        for t in range(len(te)-1):
            j=int(np.argmin(np.abs(e.EXPO-e.vt[t]))); _,_,d=e.step(j)
        rows.append(dict(market=m,group=GROUP[m],arm=arm,agent="VTrule",state="-",seed=-1,
                         **perf(np.array(e.worth)))); done.add(("VTrule","-",-1)); save()
    t0=time.time()
    for ag in CFG["agents"]:
        for kind in states:
            for sd_ in CFG["seeds"]:
                if (ag,kind,sd_) in done: continue
                etr=mk(tr,kind,slice(0,iva),arm=="B")
                ete=mk(te,kind,slice(iva,None),arm=="B")
                pol=TRAIN[ag](etr,CFG,sd_)
                mres=perf(rollout(pol,ete))
                rows.append(dict(market=m,group=GROUP[m],arm=arm,agent=ag,state=kind,seed=sd_,**mres))
                done.add((ag,kind,sd_)); save()
                print(f"  [{m}/{arm}] {ag:10s} {kind:9s} s{sd_} -> Sharpe {mres['Sharpe']:+.3f}")
    print(f"[{m}/{arm}] done in {time.time()-t0:.0f}s -> {path}")
print("runner ready. Run the cells below (resumable).")

runner ready. Run the cells below (resumable).


In [ ]:
for _name in ["CKPT","run_market","load_market","build_features"]:
    assert _name in globals(), f"{_name} not defined — run NB5_extension.ipynb up to 'runner ready' first"
import os
print("CKPT =", CKPT)
print("nb4_outputs =", os.path.dirname(CKPT))
print("environment OK")

CKPT = /content/drive/MyDrive/nb4_outputs/checkpoints
nb4_outputs = /content/drive/MyDrive/nb4_outputs
environment OK


In [ ]:
import os, glob, shutil
MISS=["MexIPC","Vietnam","ETH"]
OUTDIR=os.path.dirname(CKPT)                     # .../nb4_outputs

def _find(market, kind):                          # kind: "price"(.csv) | "signals"(.npz)
    ext=".csv" if kind=="price" else ".npz"
    pats=[os.path.join(CKPT, f"{market}_{kind}{ext}"),
          os.path.join(OUTDIR, f"{market}_{kind}{ext}"),
          os.path.join(OUTDIR,"**",f"*{market}*{kind}*{ext}"),
          os.path.join(OUTDIR,"**",f"*{market}*{ext}")]
    for p in pats:
        hits=glob.glob(p, recursive=True)
        if "**" in p and kind!="price":
            hits=[h for h in hits if kind in os.path.basename(h).lower()]
        if hits: return sorted(hits, key=len)[0]
    return None

for m in MISS:
    fp_dst=os.path.join(CKPT,f"{m}_price.csv"); fs_dst=os.path.join(CKPT,f"{m}_signals.npz")
    fp_src=_find(m,"price"); fs_src=_find(m,"signals")
    print(f"[{m}] price   -> {fp_src}")
    print(f"[{m}] signals -> {fs_src}")
    if fp_src and os.path.abspath(fp_src)!=os.path.abspath(fp_dst): shutil.copy(fp_src,fp_dst)
    if fs_src and os.path.abspath(fs_src)!=os.path.abspath(fs_dst): shutil.copy(fs_src,fs_dst)

print("\n--- cache check ---")
ready=[]
for m in MISS:
    fp=os.path.join(CKPT,f"{m}_price.csv"); fs=os.path.join(CKPT,f"{m}_signals.npz")
    ok=os.path.exists(fp) and os.path.exists(fs)
    print(f"  {m}: price {os.path.exists(fp)} | signals {os.path.exists(fs)}")
    if ok: ready.append(m)

for m in ready:
    print("="*70,"\n",m)
    run_market(m, arm="A"); run_market(m, arm="B")

missing=[m for m in MISS if m not in ready]
if missing:
    print("\nSTILL MISSING:", missing, "-> run Cell C to list the real filenames")
else:
    print("\nALL 3 DONE — re-run the Analysis cell (Section 5) for the full 14-market panel.")

[MexIPC] price   -> /content/drive/MyDrive/nb4_outputs/checkpoints/MexIPC_price.csv
[MexIPC] signals -> /content/drive/MyDrive/nb4_outputs/checkpoints/MexIPC_signals.npz
[Vietnam] price   -> /content/drive/MyDrive/nb4_outputs/checkpoints/Vietnam_price.csv
[Vietnam] signals -> /content/drive/MyDrive/nb4_outputs/checkpoints/Vietnam_signals.npz
[ETH] price   -> /content/drive/MyDrive/nb4_outputs/checkpoints/ETH_price.csv
[ETH] signals -> /content/drive/MyDrive/nb4_outputs/checkpoints/ETH_signals.npz

--- cache check ---
  MexIPC: price True | signals True
  Vietnam: price True | signals True
  ETH: price True | signals True
 MexIPC
[MexIPC/A] resume: 31 rows
[MexIPC/A] complete, skip
[MexIPC/B] resume: 21 rows
[MexIPC/B] complete, skip
 Vietnam
[Vietnam/A] resume: 31 rows
[Vietnam/A] complete, skip
[Vietnam/B] resume: 17 rows
  [Vietnam/B] PPO        +VolFore  s1 -> Sharpe +0.169
  [Vietnam/B] PPO        +VolFore  s2 -> Sharpe +0.326
  [Vietnam/B] PPO        +VolFore  s3 -> Sharpe +0.320


In [ ]:
import os, glob
OUTDIR=os.path.dirname(CKPT)
hits=[x for x in glob.glob(os.path.join(OUTDIR,"**","*"),recursive=True)
      if any(k.lower() in os.path.basename(x).lower() for k in ["MexIPC","Vietnam","ETH"])]
for h in sorted(hits): print(h)
print("\nCopy this list back so the finder can be pointed at the exact names.")

### SP500

In [ ]:
run_market("SP500", arm="A")   # effect size
run_market("SP500", arm="B")   # residual vs benchmark

[SP500/A] resume: 23 rows
  [SP500/A] PPO        +RetFore  s2 -> Sharpe +1.127
  [SP500/A] PPO        +RetFore  s3 -> Sharpe +1.216
  [SP500/A] PPO        +RetFore  s4 -> Sharpe +1.164
  [SP500/A] PPO        +VolFore  s0 -> Sharpe +1.192
  [SP500/A] PPO        +VolFore  s1 -> Sharpe +1.043
  [SP500/A] PPO        +VolFore  s2 -> Sharpe +1.085
  [SP500/A] PPO        +VolFore  s3 -> Sharpe +1.262
  [SP500/A] PPO        +VolFore  s4 -> Sharpe +1.228
[SP500/A] done in 5099s -> /content/drive/MyDrive/nb4_outputs/ext_checkpoints/SP500_armA.csv
  [SP500/B] Q-Learning Baseline  s0 -> Sharpe +0.697
  [SP500/B] Q-Learning Baseline  s1 -> Sharpe +0.600
  [SP500/B] Q-Learning Baseline  s2 -> Sharpe +0.798
  [SP500/B] Q-Learning Baseline  s3 -> Sharpe +0.724
  [SP500/B] Q-Learning Baseline  s4 -> Sharpe +0.542
  [SP500/B] Q-Learning +VolFore  s0 -> Sharpe +0.590
  [SP500/B] Q-Learning +VolFore  s1 -> Sharpe +0.906
  [SP500/B] Q-Learning +VolFore  s2 -> Sharpe +0.499
  [SP500/B] Q-Learning +VolFore  

### FTSE100

In [ ]:
run_market("FTSE100", arm="A")   # effect size
run_market("FTSE100", arm="B")   # residual vs benchmark

  [FTSE100/A] Q-Learning Baseline  s0 -> Sharpe -0.205
  [FTSE100/A] Q-Learning Baseline  s1 -> Sharpe +0.183
  [FTSE100/A] Q-Learning Baseline  s2 -> Sharpe -0.063
  [FTSE100/A] Q-Learning Baseline  s3 -> Sharpe +0.259
  [FTSE100/A] Q-Learning Baseline  s4 -> Sharpe +0.007
  [FTSE100/A] Q-Learning +RetFore  s0 -> Sharpe +0.591
  [FTSE100/A] Q-Learning +RetFore  s1 -> Sharpe +0.349
  [FTSE100/A] Q-Learning +RetFore  s2 -> Sharpe -0.085
  [FTSE100/A] Q-Learning +RetFore  s3 -> Sharpe +0.693
  [FTSE100/A] Q-Learning +RetFore  s4 -> Sharpe +0.096
  [FTSE100/A] Q-Learning +VolFore  s0 -> Sharpe +0.612
  [FTSE100/A] Q-Learning +VolFore  s1 -> Sharpe +0.513
  [FTSE100/A] Q-Learning +VolFore  s2 -> Sharpe -0.134
  [FTSE100/A] Q-Learning +VolFore  s3 -> Sharpe +0.226
  [FTSE100/A] Q-Learning +VolFore  s4 -> Sharpe +0.173
  [FTSE100/A] PPO        Baseline  s0 -> Sharpe +1.004
  [FTSE100/A] PPO        Baseline  s1 -> Sharpe +0.928
  [FTSE100/A] PPO        Baseline  s2 -> Sharpe +1.064
  [FTSE100

### KOSPI

In [ ]:
run_market("KOSPI", arm="A")   # effect size
run_market("KOSPI", arm="B")   # residual vs benchmark

[KOSPI/A] resume: 30 rows
  [KOSPI/A] PPO        +VolFore  s4 -> Sharpe +1.102
[KOSPI/A] done in 1226s -> /content/drive/MyDrive/nb4_outputs/ext_checkpoints/KOSPI_armA.csv
  [KOSPI/B] Q-Learning Baseline  s0 -> Sharpe +0.888
  [KOSPI/B] Q-Learning Baseline  s1 -> Sharpe +0.827
  [KOSPI/B] Q-Learning Baseline  s2 -> Sharpe +1.354
  [KOSPI/B] Q-Learning Baseline  s3 -> Sharpe +0.837
  [KOSPI/B] Q-Learning Baseline  s4 -> Sharpe +1.240
  [KOSPI/B] Q-Learning +VolFore  s0 -> Sharpe +1.275
  [KOSPI/B] Q-Learning +VolFore  s1 -> Sharpe +1.007
  [KOSPI/B] Q-Learning +VolFore  s2 -> Sharpe +0.770
  [KOSPI/B] Q-Learning +VolFore  s3 -> Sharpe +0.704
  [KOSPI/B] Q-Learning +VolFore  s4 -> Sharpe +1.398
  [KOSPI/B] PPO        Baseline  s0 -> Sharpe +1.289
  [KOSPI/B] PPO        Baseline  s1 -> Sharpe +1.127
  [KOSPI/B] PPO        Baseline  s2 -> Sharpe +1.340
  [KOSPI/B] PPO        Baseline  s3 -> Sharpe +1.355
  [KOSPI/B] PPO        Baseline  s4 -> Sharpe +1.397
  [KOSPI/B] PPO        +VolFore  

### Nifty50

In [ ]:
run_market("Nifty50", arm="A")   # effect size
run_market("Nifty50", arm="B")   # residual vs benchmark

[Nifty50/A] resume: 19 rows
  [Nifty50/A] PPO        Baseline  s3 -> Sharpe +0.440
  [Nifty50/A] PPO        Baseline  s4 -> Sharpe +0.331
  [Nifty50/A] PPO        +RetFore  s0 -> Sharpe +0.331
  [Nifty50/A] PPO        +RetFore  s1 -> Sharpe +0.293
  [Nifty50/A] PPO        +RetFore  s2 -> Sharpe +0.335
  [Nifty50/A] PPO        +RetFore  s3 -> Sharpe +0.351
  [Nifty50/A] PPO        +RetFore  s4 -> Sharpe +0.356
  [Nifty50/A] PPO        +VolFore  s0 -> Sharpe +0.377
  [Nifty50/A] PPO        +VolFore  s1 -> Sharpe +0.349
  [Nifty50/A] PPO        +VolFore  s2 -> Sharpe +0.332
  [Nifty50/A] PPO        +VolFore  s3 -> Sharpe +0.337
  [Nifty50/A] PPO        +VolFore  s4 -> Sharpe +0.315
[Nifty50/A] done in 5363s -> /content/drive/MyDrive/nb4_outputs/ext_checkpoints/Nifty50_armA.csv
  [Nifty50/B] Q-Learning Baseline  s0 -> Sharpe -0.269
  [Nifty50/B] Q-Learning Baseline  s1 -> Sharpe -0.375
  [Nifty50/B] Q-Learning Baseline  s2 -> Sharpe -0.182
  [Nifty50/B] Q-Learning Baseline  s3 -> Sharpe +0

### DSE

In [ ]:
run_market("DSE", arm="A")   # effect size
run_market("DSE", arm="B")   # residual vs benchmark

[DSE/A] resume: 24 rows
  [DSE/A] PPO        +RetFore  s3 -> Sharpe -0.704
  [DSE/A] PPO        +RetFore  s4 -> Sharpe -0.733
  [DSE/A] PPO        +VolFore  s0 -> Sharpe -0.826
  [DSE/A] PPO        +VolFore  s1 -> Sharpe -0.590
  [DSE/A] PPO        +VolFore  s2 -> Sharpe -0.908
  [DSE/A] PPO        +VolFore  s3 -> Sharpe -0.634
  [DSE/A] PPO        +VolFore  s4 -> Sharpe -0.754
[DSE/A] done in 2401s -> /content/drive/MyDrive/nb4_outputs/ext_checkpoints/DSE_armA.csv
  [DSE/B] Q-Learning Baseline  s0 -> Sharpe -1.217
  [DSE/B] Q-Learning Baseline  s1 -> Sharpe -1.049
  [DSE/B] Q-Learning Baseline  s2 -> Sharpe -1.000
  [DSE/B] Q-Learning Baseline  s3 -> Sharpe -1.267
  [DSE/B] Q-Learning Baseline  s4 -> Sharpe -1.198
  [DSE/B] Q-Learning +VolFore  s0 -> Sharpe -1.231
  [DSE/B] Q-Learning +VolFore  s1 -> Sharpe -1.307
  [DSE/B] Q-Learning +VolFore  s2 -> Sharpe -1.450
  [DSE/B] Q-Learning +VolFore  s3 -> Sharpe -1.113
  [DSE/B] Q-Learning +VolFore  s4 -> Sharpe -1.186
  [DSE/B] PPO       

### BTC

In [ ]:
run_market("BTC", arm="A")   # effect size
run_market("BTC", arm="B")   # residual vs benchmark

  [BTC/A] Q-Learning Baseline  s0 -> Sharpe -0.106
  [BTC/A] Q-Learning Baseline  s1 -> Sharpe -1.285
  [BTC/A] Q-Learning Baseline  s2 -> Sharpe -0.714
  [BTC/A] Q-Learning Baseline  s3 -> Sharpe -0.452
  [BTC/A] Q-Learning Baseline  s4 -> Sharpe -0.598
  [BTC/A] Q-Learning +RetFore  s0 -> Sharpe +0.291
  [BTC/A] Q-Learning +RetFore  s1 -> Sharpe -0.548
  [BTC/A] Q-Learning +RetFore  s2 -> Sharpe -0.444
  [BTC/A] Q-Learning +RetFore  s3 -> Sharpe -0.791
  [BTC/A] Q-Learning +RetFore  s4 -> Sharpe -0.740
  [BTC/A] Q-Learning +VolFore  s0 -> Sharpe -0.697
  [BTC/A] Q-Learning +VolFore  s1 -> Sharpe -0.908
  [BTC/A] Q-Learning +VolFore  s2 -> Sharpe -0.942
  [BTC/A] Q-Learning +VolFore  s3 -> Sharpe -1.121
  [BTC/A] Q-Learning +VolFore  s4 -> Sharpe -1.163
  [BTC/A] PPO        Baseline  s0 -> Sharpe +0.040
  [BTC/A] PPO        Baseline  s1 -> Sharpe +0.074
  [BTC/A] PPO        Baseline  s2 -> Sharpe +0.026
  [BTC/A] PPO        Baseline  s3 -> Sharpe +0.060
  [BTC/A] PPO        Baseline  

NASDAQ

### FTSE100

In [ ]:
run_market("FTSE100", arm="A")   # effect size
run_market("FTSE100", arm="B")   # residual vs benchmark

[FTSE100/A] resume: 31 rows
[FTSE100/A] complete, skip
[FTSE100/B] resume: 21 rows
[FTSE100/B] complete, skip


### Nikkei

ASX200



*italicized text*

DAX

Bovespa

MexIPC

Vietnam

### ETH

In [ ]:
run_market("ETH", arm="A")   # effect size
run_market("ETH", arm="B")   # residual vs benchmark

## 5. Analysis — did the three changes work?

In [ ]:
from scipy.stats import wilcoxon
fs=sorted(glob.glob(os.path.join(EXT,"*_arm*.csv")))
if not fs: raise SystemExit("no extension results yet")
E=pd.concat([pd.read_csv(f) for f in fs],ignore_index=True)
print("markets:",sorted(E.market.unique()))

def paired(d,a,b,metric="Sharpe"):
    pv=d[d.agent.isin(CFG["agents"])].pivot_table(index=["market","agent","seed"],columns="state",values=metric)
    if a not in pv or b not in pv: return None
    pv=pv.dropna(subset=[a,b]); dd=(pv[a]-pv[b]).values
    if len(dd)<5: return None
    try: _,p=wilcoxon(dd)
    except ValueError: p=np.nan
    return dd.mean(),np.median(dd),p,len(dd),(dd>0).mean()

print("\n=== ARM A: does the DSR reward + fine grid enlarge the effect? ===")
A=E[E.arm=="A"]
for a,b in [("+VolFore","Baseline"),("+RetFore","Baseline"),("+VolFore","+RetFore")]:
    r=paired(A,a,b)
    if r: print(f"  {a:9s}-{b:9s} mean={r[0]:+.4f} med={r[1]:+.4f} p={r[2]:.4f} n={r[3]} win={r[4]:.0%}")
print("  (NB4 reference: +VolFore-Baseline = +0.121, +VolFore-+RetFore = +0.198)")

print("\n=== ARM B: does residual RL beat the rule it starts from? ===")
B=E[E.arm=="B"]
rows=[]
for m,g in B.groupby("market"):
    vt=g[g.agent=="VTrule"].Sharpe
    rec=dict(market=m,VT_rule=vt.iloc[0] if len(vt) else np.nan)
    for ag in CFG["agents"]:
        for st in ["Baseline","+VolFore"]:
            v=g[(g.agent==ag)&(g.state==st)].Sharpe
            rec[f"{ag}_{st}"]=v.mean() if len(v) else np.nan
    rows.append(rec)
R=pd.DataFrame(rows); print(R.round(3).to_string(index=False))
for c in [c for c in R.columns if c not in ("market","VT_rule")]:
    w=int((R[c]>R.VT_rule).sum()); print(f"  {c:22s} beats VT rule in {w}/{len(R)} markets")
print("\n  (NB4 reference: RL beat the VT rule in 2/14 markets)")
E.to_csv(os.path.join(OUT,"ext_results.csv"),index=False)
print("\nsaved ->",os.path.join(OUT,"ext_results.csv"))

markets: ['ASX200', 'BTC', 'Bovespa', 'DAX', 'DSE', 'FTSE100', 'KOSPI', 'MexIPC', 'NASDAQ', 'Nifty50', 'Nikkei', 'SP500']

=== ARM A: does the DSR reward + fine grid enlarge the effect? ===
  +VolFore -Baseline  mean=+0.0456 med=+0.0113 p=0.5114 n=118 win=53%
  +RetFore -Baseline  mean=-0.0343 med=-0.0213 p=0.2184 n=120 win=42%
  +VolFore -+RetFore  mean=+0.0804 med=+0.0038 p=0.2192 n=118 win=52%
  (NB4 reference: +VolFore-Baseline = +0.121, +VolFore-+RetFore = +0.198)

=== ARM B: does residual RL beat the rule it starts from? ===
 market  VT_rule  Q-Learning_Baseline  Q-Learning_+VolFore  PPO_Baseline  PPO_+VolFore
 ASX200    0.461                0.042                0.052         0.536         0.534
    BTC   -0.021               -0.290               -0.468         0.006        -0.036
Bovespa    0.761                0.253                0.360         0.627           NaN
    DAX    1.071                0.724                0.653         1.049         1.043
    DSE   -0.802            

In [3]:
from scipy.stats import wilcoxon
fs=sorted(glob.glob(os.path.join(EXT,"*_arm*.csv")))
if not fs: raise SystemExit("no extension results yet")
E=pd.concat([pd.read_csv(f) for f in fs],ignore_index=True)
print("markets:",sorted(E.market.unique()))

def paired(d,a,b,metric="Sharpe"):
    pv=d[d.agent.isin(CFG["agents"])].pivot_table(index=["market","agent","seed"],columns="state",values=metric)
    if a not in pv or b not in pv: return None
    pv=pv.dropna(subset=[a,b]); dd=(pv[a]-pv[b]).values
    if len(dd)<5: return None
    try: _,p=wilcoxon(dd)
    except ValueError: p=np.nan
    return dd.mean(),np.median(dd),p,len(dd),(dd>0).mean()

print("\n=== ARM A: does the DSR reward + fine grid enlarge the effect? ===")
A=E[E.arm=="A"]
for a,b in [("+VolFore","Baseline"),("+RetFore","Baseline"),("+VolFore","+RetFore")]:
    r=paired(A,a,b)
    if r: print(f"  {a:9s}-{b:9s} mean={r[0]:+.4f} med={r[1]:+.4f} p={r[2]:.4f} n={r[3]} win={r[4]:.0%}")
print("  (NB4 reference: +VolFore-Baseline = +0.121, +VolFore-+RetFore = +0.198)")

print("\n=== ARM B: does residual RL beat the rule it starts from? ===")
B=E[E.arm=="B"]
rows=[]
for m,g in B.groupby("market"):
    vt=g[g.agent=="VTrule"].Sharpe
    rec=dict(market=m,VT_rule=vt.iloc[0] if len(vt) else np.nan)
    for ag in CFG["agents"]:
        for st in ["Baseline","+VolFore"]:
            v=g[(g.agent==ag)&(g.state==st)].Sharpe
            rec[f"{ag}_{st}"]=v.mean() if len(v) else np.nan
    rows.append(rec)
R=pd.DataFrame(rows); print(R.round(3).to_string(index=False))
for c in [c for c in R.columns if c not in ("market","VT_rule")]:
    w=int((R[c]>R.VT_rule).sum()); print(f"  {c:22s} beats VT rule in {w}/{len(R)} markets")
print("\n  (NB4 reference: RL beat the VT rule in 2/14 markets)")
E.to_csv(os.path.join(OUT,"ext_results.csv"),index=False)
print("\nsaved ->",os.path.join(OUT,"ext_results.csv"))

markets: ['ASX200', 'BTC', 'Bovespa', 'DAX', 'DSE', 'ETH', 'FTSE100', 'KOSPI', 'MexIPC', 'NASDAQ', 'Nifty50', 'Nikkei', 'SP500', 'Vietnam']

=== ARM A: does the DSR reward + fine grid enlarge the effect? ===
  +VolFore -Baseline  mean=+0.0807 med=+0.0132 p=0.1634 n=140 win=54%
  +RetFore -Baseline  mean=-0.0751 med=-0.0295 p=0.0234 n=140 win=39%
  +VolFore -+RetFore  mean=+0.1558 med=+0.0172 p=0.0055 n=140 win=56%
  (NB4 reference: +VolFore-Baseline = +0.121, +VolFore-+RetFore = +0.198)

=== ARM B: does residual RL beat the rule it starts from? ===
 market  VT_rule  Q-Learning_Baseline  Q-Learning_+VolFore  PPO_Baseline  PPO_+VolFore
 ASX200    0.461                0.042                0.052         0.536         0.534
    BTC   -0.021               -0.290               -0.468         0.006        -0.036
Bovespa    0.761                0.253                0.360         0.627           NaN
    DAX    1.071                0.724                0.653         1.049         1.043
    DSE   